In [0]:
%run ./UTILS

In [0]:

%run ./CONFIG


In [0]:

dbutils.widgets.text("batch_control_key", "", "Batch Control Key")

BATCH_CONTROL_KEY = dbutils.widgets.get("batch_control_key").strip()
assert BATCH_CONTROL_KEY, "batch_control_key parameter is required"

print(f"  Batch Control Key: {BATCH_CONTROL_KEY}")


# -------------------------------------------------------------------------
# Pre-flight: verify Event Hub is configured for this environment
# -------------------------------------------------------------------------

assert EH_NAMESPACE is not None, (
    f"Event Hub not configured for environment '{env_label}'. "
    f"Add an entry to _EH_ENV_CONFIG in CONFIG."
)

In [0]:

# -------------------------------------------------------------------------
# Resolve Event Hub config from CONFIG variable
# Pattern: {BATCH_CONTROL_KEY}_EVENT_HUB (list of dicts)
# -------------------------------------------------------------------------

AUDIT_BATCH_KEY = f"{BATCH_CONTROL_KEY}_EVENT_HUB"
config_var_name = f"{BATCH_CONTROL_KEY}_EVENT_HUB"

if config_var_name not in globals():
    raise ValueError(
        f"No Event Hub config found for variable: '{config_var_name}'.\n"
        f"Define '{config_var_name}' in the CONFIG notebook."
    )

event_hub_configs = globals()[config_var_name]
active_configs = [cfg for cfg in event_hub_configs if cfg.get("is_active", False)]

assert active_configs, f"No active Event Hub entries found in '{config_var_name}'."

print(f"  Config Variable : {config_var_name}")
print(f"  Total Entries   : {len(event_hub_configs)}")
print(f"  Active Entries  : {len(active_configs)}")


In [0]:
# Watermark: last successful Event Hub publish timestamp
# Used in cell 6 to filter query results - only publish records newer than
# the last successful publish to avoid re-sending.
# Lazy transforms outside try; only action inside (SCPAP005)
watermark_query = (
    spark.read.table(AUDIT_TABLE)
    .filter(
        (F.col("BATCH_CTRL_KEY") == AUDIT_BATCH_KEY) &
        (F.col("JOB_STATUS") == "SUCCESS")
    )
    .agg(F.max("JOB_END_TIMESTAMP").alias("last_success"))
)

try:
    watermark_ts = watermark_query.collect()[0]["last_success"]
except Exception as e:
    watermark_ts = None
    print(f"  ⚠️ Watermark lookup failed (defaulting to full publish): {e}")

if watermark_ts:
    print(f"  Watermark       : {watermark_ts}")
else:
    print("  Watermark       : (none — publishing all records)")



In [0]:

# -------------------------------------------------------------------------
# Main loop: execute query, serialize, publish to Event Hub
# -------------------------------------------------------------------------

results_summary = []

for idx, cfg_entry in enumerate(active_configs, 1):
    entry_query = cfg_entry["query"]
    eh_name = cfg_entry["event_hub_name"]
    kafka_key = cfg_entry.get("kafka_key", BATCH_CONTROL_KEY)

    print(f"\n{'-' * 60}")
    print(f"  [{idx}/{len(active_configs)}] GOLD TO EVENT HUB")
    print(f"  Event Hub : {EH_NAMESPACE}/{eh_name}")
    print(f"  Kafka Key : {kafka_key}")
    print(f"  Query     : {entry_query[:100]}...")
    print(f"{'-' * 60}")

    job_start = datetime.now()

    try:
        # Execute query
        gold_df = spark.sql(entry_query)

        # Apply watermark filter - only publish records newer than last success
        if watermark_ts:
            ts_cols = [f.name for f in gold_df.schema.fields if str(f.dataType) == "TimestampType"]
            if "_insert_timestamp" in ts_cols:
                gold_df = gold_df.filter(F.col("_insert_timestamp") > F.lit(watermark_ts))
                print(f"  Watermark applied: _insert_timestamp > {watermark_ts}")

        # Quick empty check - avoids full .count() scan (OOM-safe)
        if gold_df.limit(1).count() == 0:
            print("  No records to publish — skipping")
            results_summary.append({
                "entry": idx, "event_hub": f"{EH_NAMESPACE}/{eh_name}",
                "status": "SUCCESS", "records": 0, "error": None,
            })
            continue

        # Capture max _insert_timestamp for watermark safety.
        # Using max data timestamp (not datetime.now()) prevents data loss
        # when upstream writes records during processing.
        try:
            if "_insert_timestamp" in gold_df.columns:
                max_gold_ts = gold_df.agg(F.max("_insert_timestamp")).collect()[0][0]
            else:
                max_gold_ts = None
        except Exception as e:
            max_gold_ts = None
            print(f"  ⚠️ Max gold timestamp lookup failed: {e}")

        # Serialize to JSON and add Kafka key column
        publish_df = (
            gold_df
            .withColumn("value", F.to_json(F.struct(*[F.col(c) for c in gold_df.columns])))
            .withColumn("key", F.lit(kafka_key).cast("string"))
            .select("key", "value")
        )

        # Persist to avoid double evaluation (count + write)
        publish_df.persist()
        record_count = publish_df.count()

        # Write to Event Hub via Kafka (reads from cache)
        write_opts = build_eh_kafka_write_options()
        (
            publish_df.write
            .format("kafka")
            .option("topic", eh_name)
            .options(**write_opts)
            .save()
        )
        publish_df.unpersist()

        job_end = datetime.now()
        elapsed = (job_end - job_start).total_seconds()
        print(f"  ✔️ Published {record_count} records in {elapsed:.2f}s")

        # Insert audit record
        audit_rec = build_audit_record(
            AUDIT_BATCH_KEY, kafka_key, BATCH_CONTROL_KEY,
            record_count, "SUCCESS", job_start, job_end,
            extra_job_stats={
                "event_hub": f"{EH_NAMESPACE}/{eh_name}",
                "kafka_key": kafka_key,
                "query_preview": entry_query[:200],
                "actual_job_end": str(job_end),
            },
        )
        # Override JOB_END_TIMESTAMP with max data timestamp for watermark safety
        if max_gold_ts:
            audit_rec["JOB_END_TIMESTAMP"] = max_gold_ts
        insert_audit_record(audit_rec, AUDIT_TABLE)

        results_summary.append({
            "entry": idx, "event_hub": f"{EH_NAMESPACE}/{eh_name}",
            "status": "SUCCESS", "records": record_count, "error": None,
        })

    except Exception as e:
        job_end = datetime.now()
        error_msg = str(e)
        print(f"  ✖️ FAILED: {error_msg}")
        traceback.print_exc()

        audit_rec = build_audit_record(
            AUDIT_BATCH_KEY, kafka_key, BATCH_CONTROL_KEY,
            0, "FAILED", job_start, job_end,
            error_message=error_msg,
        )
        insert_audit_record(audit_rec, AUDIT_TABLE)

        results_summary.append({
            "entry": idx, "event_hub": f"{EH_NAMESPACE}/{eh_name}",
            "status": "FAILED", "records": 0, "error": error_msg,
        })
        # Fail-fast: stop processing remaining entries on first failure
        break


In [0]:


# -------------------------------------------------------------------------
# Summary + notebook exit
# -------------------------------------------------------------------------

print(f"\n{'-' * 60}")
print(f"  GOLD TO EVENT HUB — COMPLETE")
print(f"  Batch Control Key: {BATCH_CONTROL_KEY}")
print(f"  Entries Processed: {len(results_summary)}")
for r in results_summary:
    print(f"    [{r['entry']}] {r['event_hub']} — {r['status']} ({r.get('records', 0)} records)")
print(f"{'-' * 60}")

overall_status = "SUCCESS" if all(r["status"] == "SUCCESS" for r in results_summary) else "FAILED"
total_records = sum(r.get("records", 0) for r in results_summary)

# Fail-fast: if any entry failed, raise so the orchestrator stops the pipeline 
# instead of silently continuing. Audit records are already written per-entry in cell 6.
if overall_status == "FAILED":
    failed_entries = [r for r in results_summary if r["status"] == "FAILED"]
    raise Exception(
        f"GOLD_EVENTHUB failed for {BATCH_CONTROL_KEY}: "
        f"{len(failed_entries)} entry/entries failed - {failed_entries}"
    )

dbutils.notebook.exit(json.dumps({
    "status": overall_status,
    "batch_control_key": BATCH_CONTROL_KEY,
    "entries_processed": len(results_summary),
    "total_records": total_records,
    "results": results_summary,
}))